In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [ ]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# Cell 1 — Imports & Config

In [4]:
from pyspark.sql import functions as F

DB_GOLD = "bupa_gold"
STORAGE = "clientdatastorage"
CONTAINER = "golddata"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD}")
print("Using Gold DB:", DB_GOLD)


Using Gold DB: bupa_gold


# Cell 2 — Load Silver Providers

In [5]:
silver_providers = spark.read.format("delta").load(
    "abfss://silverdata@clientdatastorage.dfs.core.windows.net/providers"
)

print("Silver providers rowcount:", silver_providers.count())
silver_providers.show(5)
silver_providers.printSchema()


25/12/06 22:19:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver providers rowcount: 5393


+-----------+----------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|
+-----------+----------+------------------+
|   PRV54400|         0|          labelled|
|   PRV55039|         1|          labelled|
|   PRV53872|         0|          labelled|
|   PRV53223|         0|          labelled|
|   PRV54888|         0|          labelled|
+-----------+----------+------------------+
only showing top 5 rows

root
 |-- Provider_ID: string (nullable = true)
 |-- Fraud_Flag: integer (nullable = true)
 |-- Fraud_Label_Source: string (nullable = true)



# Cell 3 — Add Derived Fields

We create a risk band based on Fraud_Flag.

In [6]:
dim_providers = (
    silver_providers
    .withColumn(
        "Provider_Risk_Tier",
        F.when(F.col("Fraud_Flag") == 1, "High risk")
         .otherwise("Low risk")
    )
)

dim_providers.show(5)


+-----------+----------+------------------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|Provider_Risk_Tier|
+-----------+----------+------------------+------------------+
|   PRV54400|         0|          labelled|          Low risk|
|   PRV55039|         1|          labelled|         High risk|
|   PRV53872|         0|          labelled|          Low risk|
|   PRV53223|         0|          labelled|          Low risk|
|   PRV54888|         0|          labelled|          Low risk|
+-----------+----------+------------------+------------------+
only showing top 5 rows



# Cell 4 — Final Schema & DQ

In [7]:
print("Rowcount:", dim_providers.count())
dim_providers.describe().show()


Rowcount: 5393
+-------+-----------+-------------------+------------------+------------------+
|summary|Provider_ID|         Fraud_Flag|Fraud_Label_Source|Provider_Risk_Tier|
+-------+-----------+-------------------+------------------+------------------+
|  count|       5393|               5393|              5393|              5393|
|   mean|       NULL|0.09382532913035416|              NULL|              NULL|
| stddev|       NULL|0.29161259393992145|              NULL|              NULL|
|    min|   PRV51001|                  0|          labelled|         High risk|
|    max|   PRV57763|                  1|          labelled|          Low risk|
+-------+-----------+-------------------+------------------+------------------+



# Cell 5 — Define ADLS Path

In [8]:
DIM_PROVIDERS_PATH = (
    f"abfss://{CONTAINER}@{STORAGE}.dfs.core.windows.net/dim_providers"
)

print("Writing dim_providers to:", DIM_PROVIDERS_PATH)


Writing dim_providers to: abfss://golddata@clientdatastorage.dfs.core.windows.net/dim_providers


# Cell 6 — Write to ADLS (Delta)

In [9]:
(dim_providers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DIM_PROVIDERS_PATH)
)

print("Saved Delta files for dim_providers.")


Saved Delta files for dim_providers.


# Cell 7 — Register in Hive Metastore

In [10]:
spark.sql("DROP TABLE IF EXISTS bupa_gold.dim_providers")

spark.sql(f"""
CREATE TABLE bupa_gold.dim_providers
USING DELTA
LOCATION '{DIM_PROVIDERS_PATH}'
""")

print("Registered bupa_gold.dim_providers")


Registered bupa_gold.dim_providers


# Cell 8 — Verify Table

In [11]:
spark.table("bupa_gold.dim_providers").show(10)


+-----------+----------+------------------+------------------+
|Provider_ID|Fraud_Flag|Fraud_Label_Source|Provider_Risk_Tier|
+-----------+----------+------------------+------------------+
|   PRV54400|         0|          labelled|          Low risk|
|   PRV55039|         1|          labelled|         High risk|
|   PRV53872|         0|          labelled|          Low risk|
|   PRV53223|         0|          labelled|          Low risk|
|   PRV54888|         0|          labelled|          Low risk|
|   PRV54182|         0|          labelled|          Low risk|
|   PRV55099|         0|          labelled|          Low risk|
|   PRV51262|         0|          labelled|          Low risk|
|   PRV51665|         0|          labelled|          Low risk|
|   PRV55541|         0|          labelled|          Low risk|
+-----------+----------+------------------+------------------+
only showing top 10 rows



# Cell 9 — Metrics

In [12]:
print("Distinct Provider_ID:",
      dim_providers.select("Provider_ID").distinct().count())


Distinct Provider_ID: 5393


# Cell 10 — Completion

In [13]:
print("✅ dim_providers created successfully and ready for star-schema.")


✅ dim_providers created successfully and ready for star-schema.
